# Nhánh A sau bàn giao — khảo sát QAOA nhiều lượt có kiểm soát

B/C đã nhận mốc demo Phase 3. Notebook này dùng lại dữ liệu và Aer của notebook 05 để kiểm tra **độ ổn định giữa seed và góc khởi tạo**. Đây là đợt chuẩn bị cho khảo sát trong đề cương, chưa phải benchmark cuối cùng.

Mặc định: `p1_tight`, `p1_choice`; hai điểm khởi tạo × ba seed = **12 lượt tuần tự**. Giữ `P=3`, `p=1`, 1024 shots, `maxiter=35` để biết yếu tố nào thay đổi. Sau đó chạy riêng `p0_reference` bằng cùng thiết kế (6 lượt).

Không tăng số qubit, không đổi mô hình, không lấy nghiệm tham chiếu làm kết quả QAOA. Kernel: Conda `nt409-khiem`, Python 3.12. **Restart Kernel** trước mỗi đợt; không chạy song song notebook 05 và 06.

## Bước 1 — Kiểm tra môi trường và nạp mã đã có

Dùng `src/qaoa_experiment.py` bản Aer đã bàn giao. Không cần cài thêm thư viện so với notebook 05. Nếu đã sửa module, khởi động lại kernel trước khi import.

Bảng phiên bản xác định môi trường sinh kết quả. Cùng seed nhưng khác bộ mô phỏng/phiên bản không bắt buộc cho số đo giống nhau.

In [11]:
import gc
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import scipy
import qiskit
import qiskit_aer
import qiskit_optimization

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "qaoa_experiment.py").is_file()
             and (p / "data" / "instances").is_dir()), None)
assert ROOT is not None, "Mở notebook bên trong repo nt409-task-placement."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.qubo import build_qubo
from src.qaoa_experiment import (
    prepare_reference, probe_qaoa_resources,
    run_qaoa_experiment, save_experiment_report,
)

print("Python:", sys.version.split()[0])
print("Qiskit / Aer / Optimization:",
      qiskit.__version__, qiskit_aer.__version__, qiskit_optimization.__version__)
print("Repo:", ROOT)
print("RAM khả dụng (GiB):", round(psutil.virtual_memory().available / 2**30, 2))
assert sys.version_info[:2] == (3, 12), "Chọn kernel Conda Python 3.12 của dự án."

Python: 3.12.14
Qiskit / Aer / Optimization: 2.5.2 0.17.2 0.7.0
Repo: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
RAM khả dụng (GiB): 13.12


## Bước 2 — Chốt ma trận thí nghiệm trước khi xem kết quả

- Seed `[42, 43, 44]` tạo ba lượt cho mỗi điểm khởi tạo. Seed hiện dùng cho lấy mẫu và biên dịch mạch; các mẫu hữu hạn dẫn tới đường tối ưu khác nhau.
- `goc_cu=[0.3,0.7]` là cấu hình đã dùng; `gamma_nho=[0.3,0.05]` là một điểm khởi tạo bổ sung để thử vùng góc khác. **0.05 không phải góc tối ưu đã được chứng minh**, không phải nghiệm của instance và không bảo đảm tốt hơn.
- Với p=1, thứ tự góc là `[beta_0, gamma_0]`. Chỉ đổi seed không đồng nghĩa đổi góc khởi tạo; ở đây khảo sát cả hai yếu tố và tổng hợp riêng từng điểm.
- Mỗi lượt vẫn dùng cùng số shots và ngân sách COBYLA. Không tự tăng `maxiter` riêng cho lượt khó trong cùng đợt.

Lần đầu giữ hai instance mặc định. Đợt tiếp theo đổi **chỉ** `RUN_INSTANCE_IDS` thành `["p0_reference"]`, để `RESUME_MANIFEST=None` và chạy lại từ Bước 2. Dùng ID riêng cho đợt mới. `p1_infeasible` có thể chọn làm đối chứng, nhưng không kỳ vọng tìm placement hợp lệ.

Nếu chỉ tiếp tục một đợt bị gián đoạn, đặt `RESUME_MANIFEST` bằng đường dẫn chỉ mục đã in ở Bước 5. Cấu hình và dữ liệu phải giống đợt đó.

In [12]:
# RUN_INSTANCE_IDS = ["p1_tight", "p1_choice"]
RUN_INSTANCE_IDS = ["p0_reference"]
RESUME_MANIFEST = None
SEEDS = [42, 43, 44]
INITIAL_POINTS = {"goc_cu": [0.3, 0.7], "gamma_nho": [0.3, 0.05]}
PENALTY = 3
BASE_CONFIG = {
    "shots": 1024, "reps": 1, "maxiter": 35,
    "max_qubits": 23, "max_seconds": 300,
    "aer_threads": 4, "aer_memory_mb": 2048, "min_available_gib": 2.0,
}
# RESUME_MANIFEST = None
# Ví dụ: "data/reference_results/phase4a_runs/campaign_YYYYMMDDTHHMMSSffffffZ.json"

ALLOWED_IDS = {"p1_tight", "p1_choice", "p1_infeasible", "p0_reference"}
assert RUN_INSTANCE_IDS and len(set(RUN_INSTANCE_IDS)) == len(RUN_INSTANCE_IDS)
assert set(RUN_INSTANCE_IDS) <= ALLOWED_IDS
assert SEEDS and len(set(SEEDS)) == len(SEEDS)
assert all(type(seed) is int and seed >= 0 for seed in SEEDS)
assert BASE_CONFIG["reps"] == 1 and PENALTY == 3, "Đợt này cố định p=1, P=3."
assert INITIAL_POINTS and all(len(v) == 2 and np.isfinite(v).all()
                              for v in INITIAL_POINTS.values())

instances = {name: json.loads((ROOT / "data" / "instances" / f"{name}.json")
                             .read_text(encoding="utf-8"))
             for name in RUN_INSTANCE_IDS}
assert all(item["instance_id"] == name for name, item in instances.items())

def canonical_hash(value):
    text = json.dumps(value, sort_keys=True, ensure_ascii=False,
                      separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

bundles = {name: build_qubo(item, penalty=PENALTY) for name, item in instances.items()}
references = {name: prepare_reference(bundle) for name, bundle in bundles.items()}
DESIGN = {
    "instance_ids": RUN_INSTANCE_IDS, "seeds": SEEDS,
    "initial_points": INITIAL_POINTS, "penalty": PENALTY,
    "base_config": BASE_CONFIG,
    "instance_sha256": {name: canonical_hash(item) for name, item in instances.items()},
}
DESIGN_HASH = canonical_hash(DESIGN)
planned = []
for name, init_id, seed in product(RUN_INSTANCE_IDS, INITIAL_POINTS, SEEDS):
    config = {**BASE_CONFIG, "seed": seed, "initial_point": INITIAL_POINTS[init_id].copy()}
    key = canonical_hash({"instance_sha256": DESIGN["instance_sha256"][name],
                          "penalty": PENALTY, "initial_point_id": init_id, "config": config})
    planned.append({"run_key": key, "instance_id": name,
                    "initial_point_id": init_id, "seed": seed, "config": config})
print("Thiết kế:", DESIGN_HASH)
print("Số lượt dự kiến:", len(planned))
display(pd.DataFrame([{k: row[k] for k in ["instance_id", "initial_point_id", "seed"]}
                      for row in planned]))

Thiết kế: 2304b21c3126296d37b0dbd9388680b865feea4315a23a956be83bdec520559c
Số lượt dự kiến: 6


,instance_id,initial_point_id,seed
0,p0_reference,goc_cu,42
1,p0_reference,goc_cu,43
2,p0_reference,goc_cu,44
3,p0_reference,gamma_nho,42
4,p0_reference,gamma_nho,43
5,p0_reference,gamma_nho,44


## Bước 3 — Đọc kích thước và đáp án tham chiếu

Đáp án chuẩn tính từ mô hình gốc; chỉ dùng đối chiếu. `p1_choice` tối ưu 1 host, `p1_tight` và `p0_reference` tối ưu 2 host. Bộ vô nghiệm có mục tiêu gốc `None` nhưng QUBO vẫn có cực tiểu.

Số qubit gồm x, y và slack. Dung lượng một mảng trạng thái không phải tổng RAM của Python. Bước này không cấp phát trạng thái lượng tử 18–23 qubit.

In [13]:
size_rows = []
for name, bundle in bundles.items():
    nbits = bundle["qubo"].get_num_vars()
    assert nbits <= BASE_CONFIG["max_qubits"]
    ref = references[name]
    size_rows.append({
        "Bộ dữ liệu": name, "Task": bundle["n"], "Host": bundle["m"],
        "Qubit": nbits, "Một mảng statevector (MiB)": 16 * 2**nbits / 2**20,
        "Có nghiệm gốc": ref["instance_feasible"],
        "Số host tối ưu gốc": ref["original_best_objective"],
        "Năng lượng QUBO nhỏ nhất": ref["minimum_energy"],
        "Hợp lệ khi lấy mẫu đều (%)": 100 * ref["uniform_baseline"]["feasible_probability"],
    })
display(pd.DataFrame(size_rows))

,Bộ dữ liệu,Task,Host,Qubit,Một mảng statevector (MiB),Có nghiệm gốc,Số host tối ưu gốc,Năng lượng QUBO nhỏ nhất,Hợp lệ khi lấy mẫu đều (%)
0,p0_reference,4,2,23,128.0,True,2,2.0,0.097656


## Bước 4 — Đo thử chi phí trước khi tối ưu

Mỗi instance lấy một mạch tại góc đầu của cấu hình đầu tiên. Đây là chẩn đoán tài nguyên, **không tính thành một lượt QAOA đã tối ưu** và không lưu như kết quả.

Ước tính thời gian = thời gian lấy mẫu × (`maxiter` + 1), chưa gồm mọi chi phí và không phải bảo đảm. Xem bảng trước khi chạy Bước 6. Nếu ước tính gần/vượt ngân sách hoặc RAM khả dụng thấp, dừng để điều chỉnh nguồn lực/thiết kế.

Aer chạy tuần tự, tối đa 4 luồng cho một mạch. Giới hạn 2048 MiB chỉ áp cho lưu trạng thái của Aer, không giới hạn cứng toàn bộ Python. RAM đỉnh là đỉnh toàn vòng đời tiến trình, có thể gồm các cell trước.

In [14]:
probes = {}
for name in RUN_INSTANCE_IDS:
    trial = next(row for row in planned if row["instance_id"] == name)
    print("Đo thử:", name, flush=True)
    probes[name] = probe_qaoa_resources(bundles[name], trial["config"])
probe_rows = [{
    "Bộ dữ liệu": name, "Qubit": p["num_qubits"],
    "Một lần lấy mẫu (s)": p["sampling_seconds"],
    "Ước tính phần lấy mẫu/lượt (s)": p["rough_maxiter_plus_final_sampling_seconds"],
    "RAM đỉnh Python (MiB)": p["resources"]["process_peak_rss_mib"],
    "RAM khả dụng sau đo (GiB)": p["resources"]["system_available_after_gib"],
} for name, p in probes.items()]
display(pd.DataFrame(probe_rows))
rough_campaign = sum(probes[row["instance_id"]]["rough_maxiter_plus_final_sampling_seconds"]
                     for row in planned)
print("Ước tính riêng phần lấy mẫu của đợt (phút):", round(rough_campaign / 60, 2))
if any(p["rough_maxiter_plus_final_sampling_seconds"] >= BASE_CONFIG["max_seconds"]
       for p in probes.values()):
    raise RuntimeError("Đo thử đã chạm ngân sách: xem tài nguyên trước khi chạy Bước 6.")

Đo thử: p0_reference


,Bộ dữ liệu,Qubit,Một lần lấy mẫu (s),Ước tính phần lấy mẫu/lượt (s),RAM đỉnh Python (MiB),RAM khả dụng sau đo (GiB)
0,p0_reference,23,0.519896,18.716274,236.941406,13.741314


Ước tính riêng phần lấy mẫu của đợt (phút): 1.87


## Bước 5 — Chuẩn bị chỉ mục hoặc tiếp tục đợt đã có

Một JSON lượt chạy là một thí nghiệm. File `campaign_...json` chỉ mô tả thiết kế, các lượt hoàn tất và đường dẫn; không phải lượt bổ sung.

Chạy lại **riêng cell Bước 6** sẽ bỏ qua những lượt đã hoàn tất trong đợt hiện tại. Sau khi khởi động lại kernel, dùng `RESUME_MANIFEST` ở Bước 2 để tiếp tục đúng đợt. Đầu vào/cấu hình khác sẽ bị từ chối để tránh trộn các thiết kế.

Chỉ mục được cập nhật ngay sau mỗi lượt hoàn tất. Lượt chưa hoàn tất không có JSON kết quả riêng; số lượt còn lại là “chưa có báo cáo hoàn tất”, không tự diễn giải là vô nghiệm hay đã chạy thất bại.

In [15]:
if RESUME_MANIFEST is None:
    campaign_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    manifest_path = ROOT / "data" / "reference_results" / "phase4a_runs" / f"campaign_{campaign_id}.json"
    completed_rows = []
else:
    manifest_path = ROOT / RESUME_MANIFEST
    saved_campaign = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert saved_campaign["scope"] == "phase4a_repeated_qaoa_pilot"
    assert saved_campaign["design_hash"] == DESIGN_HASH, "Cấu hình hoặc input đã khác đợt cần tiếp tục."
    campaign_id = saved_campaign["campaign_id"]
    completed_rows = saved_campaign["completed_runs"]
    assert len({row["run_key"] for row in completed_rows}) == len(completed_rows)
    expected = {row["run_key"] for row in planned}
    for row in completed_rows:
        assert row["run_key"] in expected
        path = ROOT / row["report"]
        old = json.loads(path.read_text(encoding="utf-8"))
        assert old["status"] == "completed"
        assert old["experiment"]["campaign_id"] == campaign_id
        assert old["experiment"]["design_hash"] == DESIGN_HASH
        assert old["experiment"]["run_key"] == row["run_key"]
        assert old["instance_sha256"] == DESIGN["instance_sha256"][row["instance_id"]]
        current_versions = {
            "python_version": platform.python_version(), "numpy_version": np.__version__,
            "scipy_version": scipy.__version__, "qiskit_version": qiskit.__version__,
            "qiskit_aer_version": qiskit_aer.__version__,
            "qiskit_optimization_version": qiskit_optimization.__version__,
        }
        for key, version in current_versions.items():
            assert old[key] == version, f"Phiên bản {key} đã khác đợt cần tiếp tục."
        assert old["platform"] == platform.platform(), "Môi trường hệ điều hành đã khác đợt cần tiếp tục."
        del old

print("Đợt:", campaign_id)
print("Chỉ mục:", manifest_path.relative_to(ROOT).as_posix())
print("Hoàn tất / dự kiến:", len(completed_rows), "/", len(planned))

def save_campaign():
    completed_keys = {row["run_key"] for row in completed_rows}
    payload = {
        "schema_version": 1, "scope": "phase4a_repeated_qaoa_pilot",
        "campaign_id": campaign_id, "design_hash": DESIGN_HASH, "design": DESIGN,
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "planned_run_count": len(planned), "completed_run_count": len(completed_rows),
        "planned_runs": [{k: row[k] for k in ["run_key", "instance_id", "initial_point_id", "seed"]}
                         for row in planned],
        "completed_runs": completed_rows,
        "remaining_run_keys": [row["run_key"] for row in planned if row["run_key"] not in completed_keys],
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = manifest_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
                         encoding="utf-8")
    temporary.replace(manifest_path)

Đợt: 20261005T043159632930Z
Chỉ mục: data/reference_results/phase4a_runs/campaign_20261005T043159632930Z.json
Hoàn tất / dự kiến: 0 / 6


## Bước 6 — Chạy các lượt tuần tự và lưu ngay

Mỗi cặp seed/góc dùng một QAOA mới, cùng ngân sách. Mỗi lượt hoàn tất được lưu theo định dạng `metrics` của notebook 05, bổ sung `experiment` để liên kết với đợt này. Thư mục báo cáo vẫn là `phase3_runs/`, giúp C dùng bộ đọc đã có.

Chỉ giữ các hàng tóm tắt trong RAM; mẫu đầy đủ nằm trên đĩa. Giải phóng đối tượng mỗi lượt để tránh giữ nhiều báo cáo/mạch trong bộ nhớ.

`completed` chưa bảo đảm có nghiệm hợp lệ; 0/1024 vẫn được lưu đúng. Nếu chạm giới hạn thời gian, dừng đợt, không lưu JSON của lượt đó. Giới hạn kiểm tra sau một lần đánh giá, không ngắt giữa mạch. Các lượt hoàn tất trước đó vẫn còn và có thể tiếp tục qua chỉ mục.

In [16]:
completed_keys = {row["run_key"] for row in completed_rows}
for trial in planned:
    if trial["run_key"] in completed_keys:
        print("Bỏ qua lượt đã hoàn tất:", trial["instance_id"], trial["initial_point_id"], trial["seed"])
        continue
    name = trial["instance_id"]
    print(f"START {name} / {trial['initial_point_id']} / seed={trial['seed']}", flush=True)
    report = run_qaoa_experiment(bundles[name], trial["config"], references[name])
    if report["status"] != "completed":
        print("Dừng đợt:", report["stop_reason"], "Không lưu JSON lượt này.")
        del report
        gc.collect()
        break
    report["experiment"] = {
        "campaign_id": campaign_id, "design_hash": DESIGN_HASH,
        "run_key": trial["run_key"], "initial_point_id": trial["initial_point_id"],
        "purpose": "controlled_seed_and_initial_point_pilot",
    }
    path = save_experiment_report(ROOT, report)
    metrics = report["metrics"]
    best = metrics["best_observed_valid"]
    opt_prob = metrics["optimal_placement_probability"]
    row = {
        "run_key": trial["run_key"], "instance_id": name,
        "initial_point_id": trial["initial_point_id"], "seed": trial["seed"],
        "report": path.relative_to(ROOT).as_posix(),
        "num_qubits": report["num_qubits"], "penalty": report["penalty"],
        "shots": report["config"]["shots"], "reps": report["config"]["reps"],
        "maxiter": report["config"]["maxiter"],
        "original_best_objective": report["reference"]["original_best_objective"],
        "observed_best_objective": None if best is None else best["objective"],
        "found_valid": best is not None,
        "found_optimal": None if opt_prob is None else bool(opt_prob > 0),
        "feasible_probability": metrics["feasible_probability"],
        "optimal_placement_probability": opt_prob,
        "qubo_ground_state_probability": metrics["qubo_ground_state_probability"],
        "final_sample_mean_qubo_energy": metrics["final_sample_mean_qubo_energy"],
        "most_frequent_valid": metrics["most_frequent_valid"],
        "elapsed_seconds": report["elapsed_seconds"],
        "postprocessing_seconds": report["postprocessing_seconds"],
        "cost_function_evals": report["cost_function_evals"],
        "process_peak_rss_mib": report["resources"]["process_peak_rss_mib"],
    }
    completed_rows.append(row)
    save_campaign()
    completed_keys.add(trial["run_key"])
    print(f"SAVED: {path.name}; hợp lệ={round(row['feasible_probability'] * row['shots'])}/{row['shots']}; "
          f"thời gian={row['elapsed_seconds']:.2f}s", flush=True)
    del report, metrics, best
    gc.collect()
print("Hoàn tất trong đợt:", len(completed_rows), "/", len(planned))

START p0_reference / goc_cu / seed=42


  evaluation 1: mean E=793.263672, elapsed=0.6s
  evaluation 5: mean E=773.605469, elapsed=2.7s
  evaluation 10: mean E=707.354492, elapsed=5.2s
  evaluation 15: mean E=811.536133, elapsed=7.6s
  evaluation 20: mean E=696.104492, elapsed=10.1s
  evaluation 25: mean E=685.830078, elapsed=12.5s
SAVED: p0_reference_p1_seed42_20261005T043224195341Z.json; hợp lệ=0/1024; thời gian=14.99s
START p0_reference / goc_cu / seed=43
  evaluation 1: mean E=760.580078, elapsed=0.5s
  evaluation 5: mean E=744.022461, elapsed=2.5s
  evaluation 10: mean E=726.333984, elapsed=4.9s
  evaluation 15: mean E=753.845703, elapsed=7.3s
  evaluation 20: mean E=714.823242, elapsed=9.7s
  evaluation 25: mean E=674.798828, elapsed=12.0s
SAVED: p0_reference_p1_seed43_20261005T043239230185Z.json; hợp lệ=0/1024; thời gian=13.43s
START p0_reference / goc_cu / seed=44
  evaluation 1: mean E=762.995117, elapsed=0.5s
  evaluation 5: mean E=743.887695, elapsed=2.5s
  evaluation 10: mean E=718.501953, elapsed=4.9s
  evaluati

## Bước 7 — Đọc từng lượt trước khi lấy trung bình

Mỗi hàng là một lượt, không phải một instance duy nhất. `Hợp lệ (%)` tính trên 1024 phép đo cuối. `Host quan sát tốt nhất` chỉ có số khi thực sự đo được nghiệm hợp lệ; NaN/None nghĩa là chưa có nghiệm quan sát được, không phải 0 host.

Giữ cả các hàng có 0 nghiệm. Thời gian ghi ở đây là thời gian bộ giải; giải mã được lưu riêng trong báo cáo.

In [17]:
results = pd.DataFrame(completed_rows)
if results.empty:
    print("Chưa có lượt hoàn tất để tổng hợp.")
else:
    table = results[["instance_id", "initial_point_id", "seed", "feasible_probability",
                     "optimal_placement_probability", "observed_best_objective",
                     "most_frequent_valid", "elapsed_seconds"]].copy()
    table["feasible_probability"] *= 100
    table["optimal_placement_probability"] = pd.to_numeric(table["optimal_placement_probability"]) * 100
    display(table.rename(columns={
        "instance_id": "Bộ dữ liệu", "initial_point_id": "Khởi tạo", "seed": "Seed",
        "feasible_probability": "Hợp lệ (%)", "optimal_placement_probability": "Placement tối ưu (%)",
        "observed_best_objective": "Host quan sát tốt nhất",
        "most_frequent_valid": "Mẫu phổ biến nhất hợp lệ", "elapsed_seconds": "Bộ giải (s)",
    }).round(6))

,Bộ dữ liệu,Khởi tạo,Seed,Hợp lệ (%),Placement tối ưu (%),Host quan sát tốt nhất,Mẫu phổ biến nhất hợp lệ,Bộ giải (s)
0,p0_reference,goc_cu,42,0.000000,0.000000,NaN,False,14.985394
1,p0_reference,goc_cu,43,0.000000,0.000000,NaN,False,13.425353
2,p0_reference,goc_cu,44,0.000000,0.000000,NaN,False,15.796856
3,p0_reference,gamma_nho,42,0.000000,0.000000,NaN,False,16.212563
4,p0_reference,gamma_nho,43,0.000000,0.000000,NaN,False,16.220646
5,p0_reference,gamma_nho,44,0.097656,0.097656,2.0,False,15.064120


## Bước 8 — Đánh giá ổn định theo từng điểm khởi tạo

Tách riêng mỗi instance và điểm khởi tạo; ba seed là ba lượt trong nhóm. Hai chỉ số khác nhau:

- **Tỷ lệ mẫu hợp lệ:** phần trăm phép đo hợp lệ trong một lượt; lấy trung bình và độ lệch chuẩn giữa seed.
- **Số lượt có nghiệm:** có ít nhất một mẫu hợp lệ trong lượt. Ví dụ 2/3 lượt tìm được nghiệm không có nghĩa 66.67% mẫu là hợp lệ.

Độ lệch chuẩn dùng mẫu (`ddof=1`); một lượt chưa đủ để tính. Ba seed chỉ là khảo sát nhỏ, chưa đủ kết luận thống kê mạnh. Nếu đợt thiếu lượt, ghi rõ số hoàn tất và số dự kiến. Không chỉ trích seed tốt nhất.

Mốc lấy mẫu đều là tính chính xác, không phải một lượt ngẫu nhiên chạy thêm. Nếu P thay đổi ở đợt sau, không so năng lượng thô trực tiếp giữa các P.

In [18]:
def optional_sum(series):
    return None if series.isna().all() else int(series.sum())

summary = pd.DataFrame()
if not results.empty:
    summary = results.groupby(["instance_id", "initial_point_id"], sort=False).agg(
        completed_runs=("run_key", "size"),
        valid_runs=("found_valid", "sum"),
        optimal_runs=("found_optimal", optional_sum),
        feasible_pct_mean=("feasible_probability", lambda s: 100 * s.mean()),
        feasible_pct_std=("feasible_probability", lambda s: 100 * s.std(ddof=1)),
        optimal_pct_mean=("optimal_placement_probability", lambda s: 100 * pd.to_numeric(s).mean()),
        mean_qubo_energy=("final_sample_mean_qubo_energy", "mean"),
        mean_solver_seconds=("elapsed_seconds", "mean"),
    ).reset_index()
    summary["planned_runs"] = len(SEEDS)
    summary["uniform_feasible_pct"] = summary["instance_id"].map(
        lambda name: 100 * references[name]["uniform_baseline"]["feasible_probability"])
    display(summary.round(6))
    print("Các cột *_pct là phần trăm; *_std là độ lệch chuẩn giữa các lượt.")
    print("optimal_runs/optimal_pct_mean = NaN/N/A cho bộ vô nghiệm.")
    print("Chỉ kết luận cho cấu hình và các lượt thực sự có trong đợt này.")

,instance_id,initial_point_id,completed_runs,valid_runs,optimal_runs,feasible_pct_mean,feasible_pct_std,optimal_pct_mean,mean_qubo_energy,mean_solver_seconds,planned_runs,uniform_feasible_pct
0,p0_reference,goc_cu,3,0,0,0.000000,0.000000,0.000000,657.984375,14.735868,3,0.097656
1,p0_reference,gamma_nho,3,1,1,0.032552,0.056382,0.032552,591.646810,15.832443,3,0.097656


Các cột *_pct là phần trăm; *_std là độ lệch chuẩn giữa các lượt.
optimal_runs/optimal_pct_mean = NaN/N/A cho bộ vô nghiệm.
Chỉ kết luận cho cấu hình và các lượt thực sự có trong đợt này.


## Bước 9 — Kiểm tra mẫu thô của một lượt

Chọn chỉ số hàng trong `completed_rows`. Bảng dưới giữ thứ tự tần suất từ báo cáo; mẫu đứng đầu có thể vi phạm. `best_observed_valid` là một cách đọc khác và phải báo cáo riêng nếu đề cương dùng mẫu phổ biến nhất.

Một placement hợp lệ có thể có slack sai, nên tỷ lệ placement hợp lệ/tối ưu và tỷ lệ cực tiểu QUBO khác nhau. Không sửa bit rồi gọi đó là mẫu đo thô.

In [19]:
INSPECT_RUN_INDEX = 0
if completed_rows:
    entry = completed_rows[INSPECT_RUN_INDEX]
    inspected = json.loads((ROOT / entry["report"]).read_text(encoding="utf-8"))
    samples = inspected["final_samples"]
    assert abs(sum(item["probability"] for item in samples) - 1) < 1e-6
    print("Nguồn:", entry["report"])
    print("Khởi tạo:", entry["initial_point_id"], "seed:", entry["seed"])
    print("Nghiệm hợp lệ tốt nhất thực sự quan sát:", inspected["metrics"]["best_observed_valid"])
    display(pd.DataFrame([{
        "Xác suất": item["probability"], "Năng lượng": item["energy"],
        "Hợp lệ": item["decoded"]["feasible"], "Host": item["decoded"]["objective"],
        "x": item["decoded"]["x"], "y": item["decoded"]["y"],
    } for item in samples[:5]]))
    del inspected, samples
    gc.collect()

Nguồn: data/reference_results/phase3_runs/p0_reference_p1_seed42_20261005T043224195341Z.json
Khởi tạo: goc_cu seed: 42
Nghiệm hợp lệ tốt nhất thực sự quan sát: None


,Xác suất,Năng lượng,Hợp lệ,Host,x,y
0,0.000977,742.0,False,None,"[[1, 1], [0, 1], [0, 1], [1, 0]]","[1, 0]"
1,0.000977,952.0,False,None,"[[1, 1], [0, 0], [1, 1], [1, 0]]","[1, 0]"
2,0.000977,681.0,False,None,"[[0, 1], [0, 1], [1, 0], [0, 0]]","[0, 0]"
3,0.000977,493.0,False,None,"[[1, 1], [1, 1], [0, 1], [0, 0]]","[1, 0]"
4,0.000977,226.0,False,None,"[[1, 0], [0, 1], [0, 1], [0, 1]]","[0, 1]"


## Bước 10 — Kiểm tra gói gửi C

Đối chiếu bảng tóm tắt với JSON nguồn để không nhầm lượt hoặc mất file. Báo cáo vẫn có `schema_version=2`, `metrics`, `reference`, `final_samples` như notebook 05; chỉ bổ sung `experiment`. Chỉ mục có schema riêng và không được cộng vào số lượt.

Gửi C notebook có đầu ra, chỉ mục và toàn bộ JSON được chỉ mục tham chiếu. Không yêu cầu C chạy lại QAOA.

In [20]:
if completed_rows:
    assert manifest_path.is_file()
    saved_campaign = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert saved_campaign["completed_run_count"] == len(completed_rows)
    assert len({item["run_key"] for item in completed_rows}) == len(completed_rows)
    for entry in completed_rows:
        source = json.loads((ROOT / entry["report"]).read_text(encoding="utf-8"))
        assert source["status"] == "completed"
        assert source["experiment"]["run_key"] == entry["run_key"]
        assert source["experiment"]["design_hash"] == DESIGN_HASH
        assert source["instance_sha256"] == DESIGN["instance_sha256"][entry["instance_id"]]
        for key in ["feasible_probability", "optimal_placement_probability",
                    "qubo_ground_state_probability", "most_frequent_valid"]:
            assert source["metrics"][key] == entry[key]
        del source
    print("PASS — các lượt duy nhất, báo cáo nguồn và chỉ mục khớp nhau.")
    print("Gửi C chỉ mục:", manifest_path.relative_to(ROOT).as_posix())
    print("Số hoàn tất / dự kiến:", len(completed_rows), "/", len(planned))
else:
    print("Chưa có lượt hoàn tất; không có báo cáo hoặc chỉ mục kết quả để bàn giao.")

PASS — các lượt duy nhất, báo cáo nguồn và chỉ mục khớp nhau.
Gửi C chỉ mục: data/reference_results/phase4a_runs/campaign_20261005T043159632930Z.json
Số hoàn tất / dự kiến: 6 / 6


## Sau đợt chạy

1. Lưu notebook có đầu ra. Gửi bảng Bước 7–8 và chỉ mục cho C.
2. Nếu đã chạy hai instance mặc định, mở đợt riêng `RUN_INSTANCE_IDS=["p0_reference"]`, giữ các thông số khác, `RESUME_MANIFEST=None`, chạy từ Bước 2. Có thể Restart Kernel và chạy lại toàn bộ để đo RAM từ đầu.
3. Nếu p0_reference vẫn có 0 nghiệm hợp lệ, ghi nhận kết quả đó. Nhiều lượt cải thiện cơ hội khám phá nhưng không bảo đảm có nghiệm.
4. Bước tiếp theo mới là khảo sát P, rồi p=1/2/3. Tách các yếu tố để giải thích được thay đổi. Nếu mọi lượt p=1 đều kém, có căn cứ chuyển sang khảo sát sâu hơn, không sửa dữ liệu/chép đáp án tham chiếu.
5. Đây là phần việc của A trong lúc B/C làm song song; chưa xác nhận benchmark, baseline B hoặc báo cáo C đã hoàn thành.